E-15: the Riesz weighted estimator and undersmoothing (H15-Suf, H15-Bias)

Design: `doc/2026-10-07_experiment_registration.md` (parent repository), E-15, §1.3, §1.4 (AutoDML-lasso) and §1.5–§1.9; the choices the design leaves open are the "E-15 の" rows of `docs/spec/02_data-and-experiments.md` §2.1. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-15 stage0 13_E15_rw_undersmoothing.ipynb`
- `... E-15 stage0.5 ...`, then `... E-15 stage1 ...` (each after the previous stage is committed).

Part A (deterministic) and the Part B population predictions are computed at Stage 0. DGP, rules, the replication function and the aggregation are in `grrexp/e15.py`. A solver that raises stops the notebook (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e12, e15, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(15, STAGE, cells=None if STAGE == "stage0" else e15.CELLS,
                          arms=None if STAGE == "stage0" else [a.replace("|", "-") for a in e15.ARM_LABELS])
STAGE

'stage0'

# Stage 0: Part A and the Part B population predictions (§1.9)

In [2]:
if STAGE == "stage0":
    s0, stage0_warnings, _ = baselines.run_recording_warnings(e15.stage0)
    if not s0["part_a_ok"]:
        raise RuntimeError("Part A: symbolic and floating-point values differ by more than 1e-12")
    rec.write_stage0("e15_population", s0)
    for g in s0["generators"]:
        print(g["generator"], g["certified"], g["checks"], g.get("rho_s"), g.get("cancellation"), g.get("tau_pred"))
    rec.finalize(R=None, n=None, warnings=int(sum(stage0_warnings.values())), failures=0)

SQ True {'i_gradient': True, 'i_beta_diff': True, 'ii_iv_positive_definite': True, 'iii_margin': True, 'irrelevant_zero': True} 0.0 True None
UKL1 True {'i_gradient': True, 'i_beta_diff': True, 'ii_iv_positive_definite': True, 'iii_margin': True, 'irrelevant_zero': True} 0.0 True None


# The workflow after the replications (shared by Stage 0.5 and Stage 1)

The identity of Part B in every successful RW_full fit (stops on a violation), the §1.5 metrics, the family-1 MCSE intervals, and the families H15-Suf and H15-Bias (Holm 0.01 each).

Bootstrap call order (stream 4, one generator per family). Family 1 (`e12.family1_mcse`): the rows of the summary in order (cells in order, then `e15.ARM_LABELS`), within a row CI length (mean), max weight (median), SE ratio; rows with fewer than two successes consume no draws. Family 2 (the se of the H15-Bias standardized bias): the generators SQ, UKL1 (those without cancellation), then $n = 2000, 4000, 8000$, one call each on the successful replications of L_th RW_full.

In [3]:
def load_stage0():
    return json.loads((outputs.stage_dir(15, "stage0") / "stage0_e15_population.json").read_text())


def aggregate(raw, s0, seeds):
    anem = e15.anem_check(raw)
    summ = e15.summarise(raw, s0)
    summ = summ.assign(**e12.family1_mcse(raw, summ, seeds))
    verdicts, tests, unavailable = e15.families(summ, raw, seeds)
    summ["family_verdicts"] = json.dumps({k: (v.sentence() if v else None) for k, v in verdicts.items()})
    summ["family_tests"] = json.dumps(tests)
    summ["family_rejected"] = json.dumps({k: (list(v.rejected) if v else []) for k, v in verdicts.items()})
    summ["unavailable_tests"] = json.dumps(unavailable)
    summ["anem_identity"] = json.dumps(anem)
    return summ

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

In [4]:
if STAGE == "stage0.5":
    import matplotlib.pyplot as plt

    parallel.configure_worker()
    s0 = load_stage0()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e15.CELLS):
            t0 = time.perf_counter()
            tasks = e15.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, cells=[ci])
            res = parallel.run_tasks(e15.replicate, tasks, workers)
            aggregate(e15.raw_frame(tasks, res), s0, Seeds(15, entropy=PILOT_ENTROPY))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e15.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    aggregate(raw, s0, Seeds(15, entropy=PILOT_ENTROPY)).to_csv(buf, index=False)
    buf.seek(0)
    S = pd.read_csv(buf)
    e15.tables(S, s0["part_a"])
    plt.close(e15.figure(S))
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e15.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].str.replace("|", "-", regex=False),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e15.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={"partB": outputs.PILOT_REPS}, n={"partB": list(e15.N_VALUES)},
                 warnings=e15.total_warnings(raw), failures=int((raw["status"] != "ok").sum()),
                 extra={"whole_pilot_aggregation_table_and_figure_seconds": whole_seconds})
    del raw

# Stage 1: Monte Carlo (6 workers: other runs share the machine; the results do not depend on the number of workers, as Stage 0.5 checks)

In [5]:
if STAGE == "stage1":
    parallel.configure_worker()
    tasks = e15.tasks_for(CONFIRMATORY_ENTROPY, e15.R)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e15.replicate, tasks, 6)
    print("seconds", time.perf_counter() - t0)
    raw = e15.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e15.total_warnings(raw)
    raw.shape, n_warn

In [6]:
if STAGE == "stage1":
    s0 = load_stage0()
    summ = aggregate(raw, s0, Seeds(15))  # the Part B identity stops the notebook on a violation
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("anem_identity", "family_verdicts", "family_rejected", "unavailable_tests")}

# Tables and figure (from summary.csv and the Stage 0 Part A rows only, §4)

In [7]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    for name, tex in e15.tables(S, s0["part_a"]).items():
        rec.write_table(name, tex)
    rec.save_figure(e15.figure(S), "fig_E15_coverage")
    rec.finalize(R={"partB": e15.R}, n={"partB": list(e15.N_VALUES)}, warnings=int(n_warn),
                 failures=int((S["R"] - S["R_s"]).sum()))